# 自编码器 Autoencoders

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

自编码器是一种无监督学习的神经网络，用于学习数据的压缩表示。它的目标是将输入编码到低维潜在空间，然后再重建回原始输入。

Autoencoders are unsupervised neural networks that learn compressed representations of data. The goal is to encode input to a low-dimensional latent space and then reconstruct it back to the original input.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/autoencoder.png" width=500>

# 概述 Overview

* **目标:**  学习数据的紧凑表示（压缩）用于降维或特征提取。
* **优点:** 
  * 无需标签即可学习有用特征
  * 可用于降维和异常检测
  * 可学习数据的内在结构
* **缺点:**
  * 潜在空间可能不连续
  * 训练不稳定
* **其他:** 
  * 变分自编码器 (VAE) 是最流行的生成模型之一
  * 用于去噪和图像重建

# 设置 Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 自动编码器模型 Autoencoder Model

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Autoencoder, self).__init__()
        
        # 编码器 Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        # 解码器 Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_reconstructed = self.decoder(z)
        return x_reconstructed
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)

# 去噪自编码器 Denoising Autoencoder

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, noise_factor=0.2):
        super(DenoisingAutoencoder, self).__init__()
        self.noise_factor = noise_factor
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def add_noise(self, x):
        noise = torch.randn_like(x) * self.noise_factor
        return x + noise
    
    def forward(self, x):
        noisy_x = self.add_noise(x)
        z = self.encoder(noisy_x)
        return self.decoder(z), z

# 变分自编码器 Variational Autoencoder

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE, self).__init__()
        
        # 编码器 Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc21 = nn.Linear(hidden_dim, latent_dim)  # 均值 mean
        self.fc22 = nn.Linear(hidden_dim, latent_dim)  # 标准差 std
        
        # 解码器 Decoder
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
    
    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc21(h), self.fc22(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar
    
    def loss_function(self, x_reconstructed, x, mu, logvar):
        # 重建损失 Reconstruction loss
        BCE = F.binary_cross_entropy(x_reconstructed, x, reduction='sum')
        # KL 散度损失 KL divergence loss
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return BCE + KLD

# 训练示例 Training Example

In [ ]:
# 加载 MNIST 数据 Load MNIST data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # 展平 Flatten
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)

# 超参数 Hyperparameters
input_dim = 784  # 28x28
hidden_dim = 256
latent_dim = 32
learning_rate = 0.001
num_epochs = 5

# 初始化模型 Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Autoencoder(input_dim, hidden_dim, latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
# 训练循环 Training loop
for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        optimizer.zero_grad()
        reconstructed = model(data)
        
        # 重构损失 Reconstruction loss
        loss = F.mse_loss(reconstructed, data)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# TODO

- 对比自编码器 Contrastive Autoencoders
- 稀疏自编码器 Sparse Autoencoders
- 具体自编码器 Concrete Autoencoders